# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Nirvik-49/Week-1-FlyRank-AI-Assignment/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

### Content Action Queue & Reason Codes
We translate validated model scores into a prioritized action queue for editorial teams. Actions are ranked by expected impact and tagged with explicit reason codes to maintain decision transparency:

* **Action 1: Content Refresh (`RC_DECAY_HIGH_CTR`)** — High historical CTR but decaying engagement due to document age.
* **Action 2: Metadata Optimization (`RC_HIGH_INTENT_LOW_CTR`)** — High query volume/intent but below-average CTR, indicating weak titles or snippets.
* **Action 3: Content Prune / Merge (`RC_LOW_INTENT_HIGH_AGE`)** — Stale content with zero conversion intent over extended windows.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# 1. Synthesize action queue data
np.random.seed(42)
n_docs = 100

doc_ids = [f"doc_{i:04d}" for i in range(1, n_docs + 1)]
query_intent_score = np.random.uniform(0.1, 0.99, size=n_docs)
hist_ctr = np.random.uniform(0.01, 0.35, size=n_docs)
doc_age_days = np.random.randint(10, 400, size=n_docs)

df_queue = pd.DataFrame({
    "doc_id": doc_ids,
    "intent_score": query_intent_score.round(3),
    "hist_ctr": hist_ctr.round(3),
    "doc_age_days": doc_age_days
})

# Assign Reason Codes and Actions
def assign_action(row):
    if row["intent_score"] > 0.70 and row["hist_ctr"] < 0.12:
        return "Metadata Optimization", "RC_HIGH_INTENT_LOW_CTR", 1
    elif row["hist_ctr"] > 0.20 and row["doc_age_days"] > 180:
        return "Content Refresh", "RC_DECAY_HIGH_CTR", 2
    elif row["intent_score"] < 0.30 and row["doc_age_days"] > 250:
        return "Prune / Merge", "RC_LOW_INTENT_HIGH_AGE", 3
    else:
        return "Maintain / Monitor", "RC_STABLE_PERFORMANCE", 4

actions, codes, priorities = zip(*df_queue.apply(assign_action, axis=1))
df_queue["recommended_action"] = actions
df_queue["reason_code"] = codes
df_queue["priority_rank"] = priorities

# Sort queue by priority rank and intent score
df_queue = df_queue.sort_values(by=["priority_rank", "intent_score"], ascending=[True, False]).reset_index(drop=True)

print("--- Top 10 Ranked Action Items ---")
print(df_queue[["doc_id", "priority_rank", "recommended_action", "reason_code", "intent_score"]].head(10).to_string(index=False))

--- Top 10 Ranked Action Items ---
  doc_id  priority_rank    recommended_action            reason_code  intent_score
doc_0012              1 Metadata Optimization RC_HIGH_INTENT_LOW_CTR         0.963
doc_0053              1 Metadata Optimization RC_HIGH_INTENT_LOW_CTR         0.936
doc_0056              1 Metadata Optimization RC_HIGH_INTENT_LOW_CTR         0.920
doc_0044              1 Metadata Optimization RC_HIGH_INTENT_LOW_CTR         0.909
doc_0074              1 Metadata Optimization RC_HIGH_INTENT_LOW_CTR         0.826
doc_0068              1 Metadata Optimization RC_HIGH_INTENT_LOW_CTR         0.814
doc_0052              1 Metadata Optimization RC_HIGH_INTENT_LOW_CTR         0.790
doc_0003              1 Metadata Optimization RC_HIGH_INTENT_LOW_CTR         0.751
doc_0076              1 Metadata Optimization RC_HIGH_INTENT_LOW_CTR         0.749
doc_0010              1 Metadata Optimization RC_HIGH_INTENT_LOW_CTR         0.730


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

### Operational Boundaries & Intended Use
* **Primary Persona:** Content Strategy & SEO Operations Teams.
* **Intended Use:** Non-autonomous decision-support queue for weekly content batch reviews.
* **Operational Limits:** Predictions are non-causal and invalid during seasonal traffic anomalies, major search algorithm updates, or brand crisis events where historical CTR signals distort.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Boundary enforcement filter
low_confidence_threshold = 0.40
df_queue["valid_recommendation"] = df_queue["intent_score"] >= low_confidence_threshold

invalid_count = (df_queue["valid_recommendation"] == False).sum()
print("--- Operational Limits Check ---")
print(f"Total Evaluated Recommendations: {len(df_queue)}")
print(f"Recommendations Out-of-Bounds (Low Confidence): {invalid_count}")
print(f"Operational Boundary Compliance: {((len(df_queue) - invalid_count) / len(df_queue)) * 100:.1f}%")

--- Operational Limits Check ---
Total Evaluated Recommendations: 100
Recommendations Out-of-Bounds (Low Confidence): 41
Operational Boundary Compliance: 59.0%


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

### Human-in-the-Loop Safeguards & The No-Go List
To protect brand equity and search index standing, automated actions must adhere to strict human review rules.

* **Mandatory Human Review:** All high-priority updates (`priority_rank == 1`) require editorial confirmation before metadata deployment.
* **The No-Go List (Never Automate):**
  1. Direct automated deletion of indexed pages.
  2. Unreviewed title modifications on high-converting revenue pages.
  3. Bulk content edits during active marketing campaigns.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Flag items requiring mandatory human sign-off
df_queue["human_review_required"] = df_queue["priority_rank"].isin([1, 3])

review_summary = df_queue.groupby("recommended_action")["human_review_required"].value_counts().unstack().fillna(0)
print("--- Human Review Requirements Summary ---")
print(review_summary)

--- Human Review Requirements Summary ---
human_review_required  False  True 
recommended_action                 
Content Refresh         23.0    0.0
Maintain / Monitor      59.0    0.0
Metadata Optimization    0.0   10.0
Prune / Merge            0.0    8.0


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

### Model Decay & Monitoring Triggers
Recommendations require continuous validation to prevent model stale-out:

* **Data Drift Trigger:** Retrain if query length distribution shifts by $>15\%$ baseline drift.
* **Performance Decay Trigger:** Retrain if monthly offline F1-score drops below $0.70$.
* **Cadence Trigger:** Schedule bi-weekly batch retraining on updated query/conversion logs.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Simulate retrain trigger check logic
current_f1 = 0.68
f1_threshold = 0.70
drift_percentage = 0.12

retrain_flags = {
    "f1_decay_trigger": current_f1 < f1_threshold,
    "drift_trigger": drift_percentage > 0.15,
}

should_retrain = any(retrain_flags.values())

print("--- Model Retrain Monitoring Status ---")
print(f"Current F1 Score: {current_f1} (Threshold: {f1_threshold})")
print(f"Measured Data Drift: {drift_percentage * 100}% (Threshold: 15%)")
print(f"Retrain Recommended: {should_retrain} (Triggers: {retrain_flags})")

--- Model Retrain Monitoring Status ---
Current F1 Score: 0.68 (Threshold: 0.7)
Measured Data Drift: 12.0% (Threshold: 15%)
Retrain Recommended: True (Triggers: {'f1_decay_trigger': True, 'drift_trigger': False})


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

### Paper Artifact Generation
We export the action queue CSV locally to `work/outputs/` (gitignored), export committed metrics JSON receipts to `work/outputs/playbook_metrics.json`, and export reusable visualizations to `work/figures/queue_summary.png`.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Ensure target directories exist
os.makedirs("work/outputs", exist_ok=True)
os.makedirs("work/figures", exist_ok=True)

# 1. Export Action Queue CSV (Local export, gitignored)
queue_csv_path = "work/outputs/action_queue.csv"
df_queue.to_csv(queue_csv_path, index=False)
print(f"Exported Queue CSV: {queue_csv_path}")

# 2. Export Metrics JSON Receipt (Committed to Git)
metrics_data = {
    "total_action_items": int(len(df_queue)),
    "actions_breakdown": df_queue["recommended_action"].value_counts().to_dict(),
    "human_review_items": int(df_queue["human_review_required"].sum()),
    "model_retrain_recommended": bool(should_retrain)
}

metrics_json_path = "work/outputs/playbook_metrics.json"
with open(metrics_json_path, "w") as f:
    json.dump(metrics_data, f, indent=2)
print(f"Exported Metrics JSON: {metrics_json_path}")

# 3. Export Summary Visualization (Committed to Git)
plt.figure(figsize=(8, 4))
df_queue["recommended_action"].value_counts().plot(kind="bar", color="skyblue", edgecolor="black")
plt.title("Content Action Queue Breakdown")
plt.xlabel("Recommended Action")
plt.ylabel("Document Count")
plt.tight_layout()

figure_path = "work/figures/queue_summary.png"
plt.savefig(figure_path, dpi=300)
plt.close()
print(f"Exported Summary Figure: {figure_path}")

Exported Queue CSV: work/outputs/action_queue.csv
Exported Metrics JSON: work/outputs/playbook_metrics.json
Exported Summary Figure: work/figures/queue_summary.png


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.